# Exercise 08 — Decoding Strategies from Scratch

In the previous notebook, `generate` did all the work for you. Now it is time to **open the black box**: you implement the decoding strategies yourself, using nothing but the model's logits, and check every implementation against Hugging Face.

All decoding strategies share the same loop:

```
repeat:
    logits = model(text so far)[last position]     # one forward pass
    modify the logits                              # temperature, top-k, top-p, penalties, bans, ...
    choose the next token(s)                       # argmax, keep the best beams, or sample
    append it
```

The model is the same in every case; the strategies only differ in the last two lines.

## What you will do
1. **Greedy search**: the basic loop, checked token by token against `generate`.
2. **The KV cache**: why `generate` is faster than your loop, and how to fix that.
3. **Beam search**: the real algorithm, with a fixed number of beams.
4. **Sampling**: temperature, top-k, top-p and min-p as functions on the logits, and a sampling loop.
5. **Your own rules inside `generate`**: a repetition penalty and a text without the letter *e*.
6. **Multiple-choice questions**: to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**1.1**, **1.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell that verifies your work automatically.

> **Heads up:** this notebook uses GPT-2 (~550 MB, cached from the previous notebooks). Everything runs on a CPU; the slowest cell, the timing in Part 2, takes about half a minute.

In [ ]:
%matplotlib inline
import time

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

MODEL_NAME = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()
model.generation_config.pad_token_id = tokenizer.eos_token_id

# We never train anything in this notebook, so we switch off gradient tracking everywhere
torch.set_grad_enabled(False)

Throughout the notebook, a text is a tensor of token ids of shape `(1, length)` (a batch with one sequence), or `(n, length)` for several sequences of the same length. Two helpers:

- `next_token_logits(ids)` runs the model and returns the logits at the **last** position of every sequence, shape `(n, vocab_size)`. That is all a decoding strategy needs from the model.
- `hf_generate(ids, **kwargs)` calls Hugging Face's `generate`, so that we can compare.

In [ ]:
def next_token_logits(ids):
    # The logits for the next token after every sequence in ids, (n, vocab_size)
    return model(ids).logits[:, -1, :]


def hf_generate(ids, **generate_kwargs):
    # Hugging Face's generate for a batch without padding
    return model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), **generate_kwargs)


def decode(ids):
    return tokenizer.decode(ids[0])


prompt_ids = tokenizer("I have a dream", return_tensors="pt")["input_ids"].to(device)
print(prompt_ids, decode(hf_generate(prompt_ids, max_new_tokens=10)))

## 1. Greedy search

At every step: compute the logits for the next token, take the **argmax**, append it, repeat.

**1.1 Implement `greedy_search`.**

Hints:
- `logits.argmax(dim=-1, keepdim=True)` gives the most likely token of every row, with shape `(n, 1)`, ready for `torch.cat(..., dim=1)`.
- A loop is all you need. (The original version of this exercise used recursion, which works for greedy search, but a loop is simpler and does not hit Python's recursion limit for long texts.)

In [ ]:
def greedy_search(ids, max_new_tokens):
    # Append max_new_tokens tokens to ids (1, length), each time the most likely one
    for _ in range(max_new_tokens):
        logits = ...   # TODO: the logits for the next token, (1, vocab_size)
        next_id = ...  # TODO: the most likely token, (1, 1)
        ids = ...      # TODO: append it
    return ids


print(decode(greedy_search(prompt_ids, 20)))

In [ ]:
# ✅ Check greedy_search against generate
mine = greedy_search(prompt_ids, 20)
theirs = hf_generate(prompt_ids, max_new_tokens=20, do_sample=False)
assert mine.shape == (1, prompt_ids.shape[1] + 20), f"expected shape (1, {prompt_ids.shape[1] + 20}), got {tuple(mine.shape)}"
assert torch.equal(mine, theirs), f"your greedy search differs from generate:\n{decode(mine)!r}\n{decode(theirs)!r}"
print("Looks good ✅")

## 2. The KV cache

Your loop runs the model on the **whole** text at every step, so it recomputes the keys and values of all earlier tokens again and again. But in a causal model, the keys and values of a token never change once it is in the text: a token only attends to the tokens *before* it (week 5).

So we can store them, the **KV cache**, and at every step feed the model only the **one new token**. Its query attends to the cached keys and values of all earlier tokens. Hugging Face models do this when you pass `use_cache=True`: the output then has a field `past_key_values` with the cache, which you pass back in the next call.

**2.1 Implement `greedy_search_cached`.**

Hints:
- The first call processes the whole prompt: `out = model(ids, use_cache=True)`. `out.logits` has shape `(1, length, vocab_size)`.
- Every later call gets only the last new token, of shape `(1, 1)`, and the cache: `model(next_id, past_key_values=out.past_key_values, use_cache=True)`.

In [ ]:
def greedy_search_cached(ids, max_new_tokens):
    # Greedy search that feeds the model only the new token at every step, using the KV cache
    out = model(ids, use_cache=True)
    for _ in range(max_new_tokens):
        next_id = ...  # TODO: the most likely next token, (1, 1)
        ids = ...      # TODO: append it
        out = ...      # TODO: run the model on the new token only, with the cache
    return ids


print(decode(greedy_search_cached(prompt_ids, 20)))

In [ ]:
# ✅ Check greedy_search_cached
assert torch.equal(greedy_search_cached(prompt_ids, 20), greedy_search(prompt_ids, 20)), "the cache should not change the result"
print("Looks good ✅")

**2.2 Run the cell below**, which times both versions for an increasing number of new tokens.

In [ ]:
lengths, times_plain, times_cached = [50, 100, 200], [], []
for n in lengths:
    start = time.time()
    greedy_search(prompt_ids, n)
    times_plain.append(time.time() - start)
    start = time.time()
    greedy_search_cached(prompt_ids, n)
    times_cached.append(time.time() - start)
    print(f"{n:>4} tokens: without cache {times_plain[-1]:.1f} s, with cache {times_cached[-1]:.1f} s")

plt.figure(figsize=(6, 3.5))
plt.plot(lengths, times_plain, marker="o", label="without cache")
plt.plot(lengths, times_cached, marker="o", label="with KV cache")
plt.xlabel("new tokens")
plt.ylabel("seconds")
plt.legend()
plt.show()

**2.3 Explain the timings.** How does the cost of generating $n$ tokens grow with $n$, with and without the cache? What does the cache cost in return? (Think of a model with 32 layers and a context of 100,000 tokens.)

---

*Your answer here:*

---

## 3. Beam search

Beam search keeps the `num_beams` most likely sequences. At every step it extends each beam by every possible token, scores all candidates by their **cumulative log-probability**, and keeps the best `num_beams` of them, across all beams.

### Why not just recurse?

Last year's version of this exercise implemented beam search recursively: from each sequence it followed the `num_beams` most likely next tokens, and in the end picked the best of all the sequences found.

**3.1 How many forward passes** does that recursive search need to generate 5 tokens with 2 beams? And 20 tokens with 5 beams? How many does real beam search need, which keeps only `num_beams` sequences at every step?

In [ ]:
def passes_recursive(length, beams):
    return ...  # TODO: the number of forward passes of the recursive search


def passes_beam(length, beams):
    return ...  # TODO: the number of forward passes of real beam search (count one pass per sequence)


for length, beams in [(5, 2), (20, 5)]:
    print(f"{length} tokens, {beams} beams: recursive {passes_recursive(length, beams):,}, beam search {passes_beam(length, beams):,}")

In [ ]:
# ✅ Check your counts
assert passes_recursive(5, 2) == 31, "the recursive search expands 1 + 2 + 4 + 8 + 16 sequences"
assert passes_beam(5, 2) == 9, "beam search runs the prompt once, then 2 sequences at each of the 4 remaining steps"
print("Looks good ✅  The recursive version is an exhaustive search of a tree that grows exponentially, not beam search.")

### Beam search, step by step

Because all beams have the same length, we can stack them into one tensor of shape `(n_beams, length)` and run the model on all of them in **one** forward pass. Then:

1. `log_probs = F.log_softmax(logits, dim=-1)` has shape `(n_beams, vocab_size)`.
2. The score of every candidate is the score of its beam plus the log-probability of the new token: `scores[:, None] + log_probs`, again `(n_beams, vocab_size)`.
3. Flatten that to one long vector and take the `num_beams` best candidates with `topk`. A flat index `i` means beam `i // vocab_size` and token `i % vocab_size`.
4. The new beams are the selected old beams with their new tokens appended.

At the first step there is only one beam (the prompt), so the first `topk` picks the `num_beams` best first tokens.

A note on **EOS**: a beam that ends (generates EOS) should be set aside as finished, and the search then continues with the others. That bookkeeping is the messy part of a real implementation. To keep yours simple, we forbid EOS by setting its log-probability to $-\infty$, and pass `bad_words_ids` to `generate` to do the same in the comparison.

**3.2 Implement `beam_search`.**

In [ ]:
def beam_search(ids, max_new_tokens, num_beams):
    # Beam search from ids (1, length). Returns all final beams (num_beams, length + max_new_tokens),
    # sorted from best to worst, and their scores (sums of log-probabilities of the new tokens), (num_beams,)
    beams = ids
    scores = torch.zeros(1, device=ids.device)
    for _ in range(max_new_tokens):
        log_probs = F.log_softmax(next_token_logits(beams), dim=-1)  # (n_beams, vocab_size)
        log_probs[:, tokenizer.eos_token_id] = float("-inf")  # no beam may end early
        vocab_size = log_probs.shape[-1]

        candidates = ...  # TODO: the score of every candidate, (n_beams, vocab_size)
        top_scores, top_indices = ...  # TODO: the num_beams best candidates of the flattened scores
        beam_indices = ...   # TODO: which beam each of them extends, (num_beams,)
        token_indices = ...  # TODO: and with which token, (num_beams,)

        beams = ...   # TODO: the new beams, (num_beams, length + 1)
        scores = top_scores
    return beams, scores


beams, scores = beam_search(prompt_ids, 10, num_beams=3)
for beam, score in zip(beams, scores):
    print(f"{score:7.3f}  {tokenizer.decode(beam)!r}")

To compare with `generate`, we set `length_penalty=0.0`. By default, Hugging Face divides the score of a finished beam by its length, to avoid a bias towards short outputs; with a length penalty of 0 the score is the plain sum of log-probabilities, as in your implementation.

In [ ]:
# ✅ Check beam_search against generate
for num_beams, n in [(1, 10), (3, 10), (5, 15)]:
    beams, scores = beam_search(prompt_ids, n, num_beams)
    assert beams.shape == (num_beams, prompt_ids.shape[1] + n), f"expected {num_beams} beams of {prompt_ids.shape[1] + n} tokens, got {tuple(beams.shape)}"
    out = hf_generate(
        prompt_ids, max_new_tokens=n, num_beams=num_beams, do_sample=False, length_penalty=0.0, early_stopping=True,
        bad_words_ids=[[tokenizer.eos_token_id]], return_dict_in_generate=True, output_scores=True,
    )
    assert torch.equal(beams[:1], out.sequences), f"{num_beams} beams: your best beam differs from generate's:\n{tokenizer.decode(beams[0])!r}\n{decode(out.sequences)!r}"
    if num_beams > 1:
        assert abs(scores[0].item() - out.sequences_scores[0].item()) < 1e-2, f"expected the score {out.sequences_scores[0]:.3f}, got {scores[0]:.3f}"
assert torch.equal(beam_search(prompt_ids, 10, 1)[0], greedy_search(prompt_ids, 10)), "beam search with 1 beam should be greedy search"
print("Looks good ✅")

**3.3 Does a wider beam find a more likely text?** Run the cell below, which compares the score (the log-probability of the 20 new tokens) of the best beam for several beam widths.

In [ ]:
widths = [1, 2, 3, 5, 10]
best_scores = []
for num_beams in widths:
    beams, scores = beam_search(prompt_ids, 20, num_beams)
    best_scores.append(scores[0].item())
    print(f"{num_beams:>2} beams: log-prob {scores[0]:7.2f}  {tokenizer.decode(beams[0, prompt_ids.shape[1]:])!r}")

**3.4 What do you see?** Does the score always improve with more beams? Is beam search guaranteed to find the most likely continuation of 20 tokens? What would that take?

---

*Your answer here:*

---

## 4. Sampling

Sampling strategies differ only in how they **modify the logits** before sampling. We write each modification as a function that takes logits of shape `(n, vocab_size)` and returns modified logits of the same shape, with $-\infty$ for tokens that may not be sampled. Hugging Face calls these functions **logits warpers** (or logits processors). They can be chained.

The temperature is the simplest one:

In [ ]:
def apply_temperature(logits, temperature):
    return logits / temperature


# A toy distribution over 5 tokens for the checks
toy_probs = torch.tensor([[0.5, 0.3, 0.1, 0.05, 0.05]])
toy_logits = toy_probs.log()
print(F.softmax(apply_temperature(toy_logits, 0.5), dim=-1))

**4.1 Implement `top_k_filter`**, which keeps the `k` largest logits of every row and sets all others to $-\infty$ (as in week 6, now for a batch).

Hint: `logits.topk(k, dim=-1).values[:, -1:]` is the `k`-th largest logit of every row, shape `(n, 1)`. `logits.masked_fill(mask, float("-inf"))` replaces the entries where `mask` is `True`.

In [ ]:
def top_k_filter(logits, k):
    # Keep the k largest logits of every row, set the others to -inf
    kth_largest = ...  # TODO: (n, 1)
    return ...         # TODO

In [ ]:
# ✅ Check top_k_filter
filtered = top_k_filter(toy_logits, 2)
assert filtered.shape == toy_logits.shape, "the shape should not change"
assert torch.isfinite(filtered).tolist() == [[True, True, False, False, False]], "keep the 2 most likely tokens"
assert torch.allclose(F.softmax(filtered, dim=-1), torch.tensor([[0.625, 0.375, 0, 0, 0]])), "after the softmax, the kept tokens are renormalised"
print("Looks good ✅")

**4.2 Implement `top_p_filter`**, which keeps the smallest set of most likely tokens whose probabilities add up to at least `p`.

Work in **sorted** order:
1. Sort the logits in descending order: `sorted_logits, sorted_indices = logits.sort(dim=-1, descending=True)`.
2. Compute the probabilities of the sorted logits and their cumulative sums.
3. A token is removed if the tokens *before* it already add up to at least `p`, i.e. if `cumulative - probs >= p`. (This way the most likely token is never removed, and the token that crosses `p` is kept.)
4. The mask is in sorted order. Bring it back to the original order with `mask.scatter(-1, sorted_indices, sorted_mask)`: the entry at position `sorted_indices[i]` gets `sorted_mask[i]`.

In [ ]:
def top_p_filter(logits, p):
    # Keep the smallest set of most likely tokens with a total probability of at least p, set the others to -inf
    sorted_logits, sorted_indices = logits.sort(dim=-1, descending=True)
    probs = ...         # TODO: the probabilities of the sorted logits
    cumulative = ...    # TODO: their cumulative sums
    sorted_mask = ...   # TODO: True for the tokens to remove, in sorted order
    mask = sorted_mask.scatter(-1, sorted_indices, sorted_mask)
    return logits.masked_fill(mask, float("-inf"))

In [ ]:
# ✅ Check top_p_filter
for p, expected in [(0.4, 1), (0.75, 2), (0.85, 3), (0.99, 5)]:
    kept = torch.isfinite(top_p_filter(toy_logits, p)).sum().item()
    assert kept == expected, f"with p={p}, the nucleus of [0.5, 0.3, 0.1, 0.05, 0.05] has {expected} tokens, you kept {kept}"
shuffled = toy_logits[:, [3, 0, 4, 2, 1]]  # the same distribution in another order
assert torch.isfinite(top_p_filter(shuffled, 0.75)).tolist() == [[False, True, False, False, True]], "the mask must be in the original order"
print("Looks good ✅")

### Min-p sampling

A more recent alternative is **min-p sampling** ([Nguyen et al., 2025](https://arxiv.org/abs/2407.01082)), which is popular in open-source LLM tools: it keeps every token whose probability is at least `min_p` times the probability of the **most likely** token. When the model is sure (top token 0.9), the threshold is high and few tokens survive; when it is unsure (top token 0.05), the threshold is low and many do.

**4.3 Implement `min_p_filter`.**

In [ ]:
def min_p_filter(logits, min_p):
    # Keep the tokens with a probability of at least min_p times the largest probability in the row
    probs = F.softmax(logits, dim=-1)
    threshold = ...  # TODO: (n, 1)
    return ...       # TODO

In [ ]:
# ✅ Check min_p_filter
for min_p, expected in [(0.9, 1), (0.5, 2), (0.15, 3), (0.05, 5)]:
    kept = torch.isfinite(min_p_filter(toy_logits, min_p)).sum().item()
    assert kept == expected, f"with min_p={min_p}, {expected} tokens of [0.5, 0.3, 0.1, 0.05, 0.05] should be kept, you kept {kept}"
print("Looks good ✅")

### The sampling loop

**4.4 Implement `sample`.** At every step, apply the temperature and then whichever filters are set, turn the logits into probabilities, and draw the next token with `torch.multinomial(probs, num_samples=1, generator=generator)`.

The `generator` is a random number generator with its own seed. Passing it explicitly makes the result reproducible without resetting the global seed. (The original version of this exercise called `torch.manual_seed(0)` *inside* the loop, so every step drew the "same" random number; and it passed the temperature on as 1 in its recursive call, so only the first token used the temperature. Avoid both.)

In [ ]:
def sample(ids, max_new_tokens, temperature=1.0, top_k=None, top_p=None, min_p=None, seed=0):
    # Sample max_new_tokens tokens after ids (1, length) with the given temperature and filters
    generator = torch.Generator(device=ids.device).manual_seed(seed)
    for _ in range(max_new_tokens):
        logits = ...  # TODO: the logits for the next token, divided by the temperature
        # TODO: apply top_k_filter, top_p_filter and min_p_filter, each only if its argument is not None
        probs = ...    # TODO
        next_id = ...  # TODO: draw one token, (1, 1)
        ids = torch.cat([ids, next_id], dim=1)
    return ids


for seed in range(3):
    print(repr(decode(sample(prompt_ids, 20, top_p=0.9, seed=seed))))

In [ ]:
# ✅ Check sample
greedy = greedy_search(prompt_ids, 15)
assert torch.equal(sample(prompt_ids, 15, top_k=1, seed=1), greedy), "with top_k=1, sampling is greedy search"
assert torch.equal(sample(prompt_ids, 15, top_p=1e-6, seed=1), greedy), "with a tiny p, sampling is greedy search"
assert torch.equal(sample(prompt_ids, 15, temperature=1e-4, seed=1), greedy), "with a tiny temperature, sampling is (practically) greedy search"
assert torch.equal(sample(prompt_ids, 15, seed=3), sample(prompt_ids, 15, seed=3)), "the same seed should give the same text"
assert not torch.equal(sample(prompt_ids, 15, seed=3), sample(prompt_ids, 15, seed=4)), "different seeds should give different texts"
print("Looks good ✅")

### How many tokens does each filter keep?

**4.5 Run the cell below.** It samples a text with top-p, and at every step it counts how many tokens top-k ($k = 50$), top-p ($p = 0.9$) and min-p ($0.1$) would keep.

In [ ]:
ids = sample(prompt_ids, 40, top_p=0.9, seed=0)
counts = {"top-k (50)": [], "top-p (0.9)": [], "min-p (0.1)": []}
logits = model(ids).logits[0, prompt_ids.shape[1] - 1 : -1]  # the logits that predicted each new token, (40, vocab_size)
counts["top-k (50)"] = torch.isfinite(top_k_filter(logits, 50)).sum(dim=-1).tolist()
counts["top-p (0.9)"] = torch.isfinite(top_p_filter(logits, 0.9)).sum(dim=-1).tolist()
counts["min-p (0.1)"] = torch.isfinite(min_p_filter(logits, 0.1)).sum(dim=-1).tolist()

new_tokens = [tokenizer.decode(t) for t in ids[0, prompt_ids.shape[1]:]]
plt.figure(figsize=(12, 4))
for name, values in counts.items():
    plt.plot(values, marker=".", label=name)
plt.yscale("log")
plt.xticks(range(len(new_tokens)), new_tokens, rotation=90)
plt.ylabel("tokens kept (log scale)")
plt.legend()
plt.show()
print(decode(ids))

**4.6 Describe the plot.** At which tokens do top-p and min-p keep many candidates, and at which only a handful? How do top-p and min-p differ in how they react to a flat distribution? Why is a single $k$ for every step a poor fit?

---

*Your answer here:*

---

## 5. Your own rules inside `generate`

You do not have to rewrite `generate` to change how it decodes. You can pass a list of your own **logits processors** with `logits_processor=LogitsProcessorList([...])`. A logits processor is a class with a `__call__(self, input_ids, scores)` method: `input_ids` is the text so far, shape `(batch_size, length)`, `scores` are the logits for the next token, shape `(batch_size, vocab_size)`, and it returns the modified scores. `generate` then applies it at every step, for greedy search, beam search and sampling alike.

### A repetition penalty

The **repetition penalty** of the CTRL model ([Keskar et al., 2019](https://arxiv.org/abs/1909.05858)), which you used in notebook 0, makes every token that already occurs in the text less likely: its logit is **divided** by the penalty $\theta$ if it is positive, and **multiplied** by $\theta$ if it is negative (in both cases it moves down).

**5.1 Implement `RepetitionPenalty`.**

Hints:
- `scores.gather(1, input_ids)` gives the score of every token that occurs in the text, shape `(batch_size, length)`.
- `torch.where(condition, a, b)` chooses between two tensors element by element.
- `scores.scatter(1, input_ids, values)` returns a copy of the scores with `values` written at the positions of the tokens in `input_ids`.

In [ ]:
class RepetitionPenalty(LogitsProcessor):
    def __init__(self, penalty):
        self.penalty = penalty

    def __call__(self, input_ids, scores):
        previous = ...   # TODO: the scores of the tokens that already occur, (batch_size, length)
        penalised = ...  # TODO: divided by the penalty if positive, multiplied by it if negative
        return ...       # TODO: the scores with the penalised values written back


prompt = tokenizer("I enjoy walking with my cute dog", return_tensors="pt")["input_ids"].to(device)
mine = hf_generate(prompt, max_new_tokens=40, logits_processor=LogitsProcessorList([RepetitionPenalty(1.3)]))
print(decode(mine))

In [ ]:
# ✅ Check RepetitionPenalty against generate's repetition_penalty
toy_scores = torch.tensor([[2.0, -2.0, 1.0, 0.5]])
penalised = RepetitionPenalty(2.0)(torch.tensor([[0, 1, 1]]), toy_scores.clone())
assert torch.allclose(penalised, torch.tensor([[1.0, -4.0, 1.0, 0.5]])), f"expected [[1, -4, 1, 0.5]], got {penalised}"
theirs = hf_generate(prompt, max_new_tokens=40, repetition_penalty=1.3)
assert torch.equal(mine, theirs), f"your penalty gives another text than generate's:\n{decode(mine)!r}\n{decode(theirs)!r}"
print("Looks good ✅")

### A lipogram

A **lipogram** is a text that avoids a letter. Georges Perec wrote a whole novel, *La Disparition* (1969), without the letter *e*, the most common letter in French as in English. Language models find this very hard: they think in tokens, not letters, and most tokens contain an *e* somewhere.

With a logits processor it becomes easy: forbid every token that contains the letter.

**5.2 Implement `NoLetter`.** In `__init__`, find the ids of all tokens in the vocabulary that contain the letter (upper or lower case); in `__call__`, set their scores to $-\infty$.

Hint: `tokenizer.decode([i])` is the text of token `i`; `len(tokenizer)` is the size of the vocabulary. Going through all 50,257 tokens takes a second or two.

In [ ]:
class NoLetter(LogitsProcessor):
    def __init__(self, letter, tokenizer):
        banned = ...  # TODO: the ids of all tokens that contain the letter, in lower or upper case
        self.banned = torch.tensor(banned)
        print(f"{len(banned):,} of {len(tokenizer):,} tokens contain {letter!r} or {letter.upper()!r}")

    def __call__(self, input_ids, scores):
        ...  # TODO: set the scores of the banned tokens to -inf
        return scores


no_e = NoLetter("e", tokenizer)
prompt = tokenizer("Once upon a time, a young girl", return_tensors="pt")["input_ids"].to(device)
torch.manual_seed(0)
output = hf_generate(prompt, max_new_tokens=60, do_sample=True, top_k=0, temperature=0.7, logits_processor=LogitsProcessorList([no_e]))
print(decode(output))

In [ ]:
# ✅ Check NoLetter
new_text = tokenizer.decode(output[0, prompt.shape[1]:])
assert "e" not in new_text.lower(), f"the generated text contains an e: {new_text!r}"
assert tokenizer.convert_tokens_to_ids("Ġthe") in no_e.banned.tolist(), "' the' should be banned"
assert tokenizer.convert_tokens_to_ids("Ġa") not in no_e.banned.tolist(), "' a' should not be banned"
print("Looks good ✅")

**5.3 Read the lipogram.** How good is it? Constrained decoding like this is used in practice for much more useful constraints. Name one or two, and explain why constraining the *decoding* is more reliable for them than asking for the constraint in the prompt.

---

*Your answer here:*

---

## 6. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 6.1. The KV cache

What does the KV cache store, and why can it be reused?

A. The logits of earlier steps, because they do not change<br>
B. The keys and values of earlier tokens, because in a causal model they do not depend on later tokens<br>
C. The attention weights of earlier steps, because the softmax is cached<br>
D. The token embeddings, because the vocabulary is fixed<br>

**Answer:** 

---

### 6.2. Beam search

Beam search with `num_beams=4` generates 10 tokens. How many candidate sequences are scored at each step (after the first), with a vocabulary of size $|V|$?

A. 4<br>
B. $4 \cdot |V|$<br>
C. $|V|^4$<br>
D. $4^{10}$<br>

**Answer:** 

---

### 6.3. Top-p

The next-token probabilities are 0.6, 0.25, 0.1, 0.05. Which tokens does top-p sampling with $p = 0.8$ keep?

A. Only the first<br>
B. The first two<br>
C. The first three<br>
D. All four<br>

**Answer:** 

---

### 6.4. Min-p

With min-p sampling and `min_p = 0.1`, the most likely next token has probability 0.4. Which tokens may be sampled?

A. Only tokens with a probability of at least 0.1<br>
B. Only tokens with a probability of at least 0.04<br>
C. The 10% most likely tokens of the vocabulary<br>
D. The tokens whose probabilities add up to 0.9<br>

**Answer:** 

---

### 6.5. Logits processors

You pass a custom logits processor to `generate` that sets the score of every token containing a digit to $-\infty$. What is the effect?

A. The model is retrained not to produce digits<br>
B. Digits are less likely, but can still appear when sampling at a high temperature<br>
C. No generated token can contain a digit, whatever the decoding strategy<br>
D. It only works with greedy decoding<br>

**Answer:** 